Q3: How do we find informed players in this trading market?
- Their trade size is unusually large relative to the population of trades
- Their trade size is large relative to their own history (if we have more player_id histories, and we see a player_id ABC did a trade that was much larger than their history, or they have no history, and they just did a trade that was +4 stdev bigger than average quantities in the past few hours / days / weeks, that's a sign).  But we don't have it - player_id and player name are NA here.
- They hit through the stack - they trade a price that is 2-3 cents through the best bid (sell 2-3 cents below best bid) or best ask (buy 2-3 cents above best bid)
- There might be some anomaly detection that we can do (use machine learning or anomaly detection algorithms on the features of price and quantity, especially relative to the order book?)
- Also, if they do a trade like this, and then the market moves a few min or hours after this, it means that they knew the info first, and then it leaked to other participants later.  This doesn't help us on that trade, but it helps us know that trader is an informed player so we should watch his trades going forward
- How do identify trades in real-time?  When the stack moves in a magnitude that is bigger than what it has historically done in that market, or if the trade size is unusually large compared to other trades in the market

- Another way to identify informed players is that their trades happen first, and then the market moves heavily in their favor.  So for example, we once again walk through the timeline, from start of the dataset to the end of the dataset, and every time we encounter a trade in the trades dataset, we analyze what happened in that market in the following minutes (we can pick any time frame, such as 10 minutes or 30 minutes afterward), and flag cases where the market moved in the same direction as that trader, especially if the trade did a large-sized trade
- The issue with this metric is that you cannot know that the trade was informed until after the market already moved in his favor, and likely against yours as a market-maker.  In terms of what you are given in real-time, the only thing that you can know about that trade is its price and size

- If you actually had more data on the players, such as the player name, and player_id, and you saw that they did an informed trade, then you can watch for their trades going forward.  Let's say that you saw player ABC sold p in a market at p = 0.6, and then 5 min later, p crashed down to 0.55.  Then you know that ABC probably has some special edge in the market (private non public information maybe), so you watch next time a trade from ABC happens.  Unfortunately, based on the data exploration that we did earlier, all of the columns for player name and player ID are NA in this dataset, so we can't use it anyway, if the data is in this format

Response to Claude:
- Agreed that we need to look at both books and trades datasets
- The most informed action is probably one where the size is unusually large, and it happens in a short time span, perhaps over multiple markets.  I also bet that aggressive_buy_flag is going to be True, because someone who knows that the event will resolve at either $0 or $1 will not care about paying up an extra $0.01 or $0.02 in bid/ask spread
- I would check this in the trades dataset first, and then the order book data set second
- Proposal: As before, let's walk through time, from start of the 6 hours to the end of the hours.  Let's look at every trade as we go along forward in time.  We keep a running record of the trades as they happen.  We can create a dataframe of trades, with columns = market, price, quantity, direction (aggressor_buy_flag = True is buy, False is sell), and timestamp.  Whenever we see a trade that is beyond let's say 3 standard deviations away from the average trade size, and especially if the price compares to the order book and it is beyond the best_bid or above best_ask, we flag it (the person is hitting through multiple prices in the stack)


- In terms of the risk, we can quantify the exact dollar risk by looking at the direction of the trade and the difference between the entry level and 0 vs. 1.  For instance, if someone bought 100 contracts at 0.7, the worst thing that could happen would be if the event resolved at 0, so they lost 0.7 on 100 contracts, which is a loss of $70 for instance.
- Beyond just the financial risk of monetary losses, this also risks leaking information into the market for other players - someone who is watching for such trades will get the information that someone informed made a bet that has a higher than normal chance of paying out. 
- There are also other issues like legal ramifications. Examples:

Regulators have made clear that trading event contracts on MNPI (material non-public information) can be prosecuted even though these aren't traditional securities. The CFTC's Enforcement Division put out a Prediction Markets Advisory in February 2026, publicly describing MNPI-based event-contract trading as conduct the CFTC can pursue as "insider trading" under CEA § 6(c)(1) and Rule 180.1. The legal theory used is misappropriation of confidential information, not classic securities insider trading, because event contracts fall under commodities law. 
Snell & Wilmer

Actual cases so far
- The YouTube editor case (Kalshi, 2025) — a trader who was a YouTube channel editor had advanced knowledge of video contents before they posted, and traded on that. Kalshi hit him with a $20,397.58 penalty (disgorgement plus fine) and a 2-year exchange suspension. 
Commodity Futures Trading Commission
Political candidate trading on his own race (Kalshi, May 2025) — a political candidate was found trading on his own candidacy, which Kalshi flagged and disciplined.
- Michele Spagnuolo / Google engineer (May 2026) — the CFTC and DOJ charged a Google employee with using material nonpublic information to trade Polymarket contracts tied to the browser's "Year in Search" lists — first case involving a private-sector company employee trading on internal work knowledge. 

### Plan for Q3 -- to review and revise before any code is written

#### What the question asks for

> What is the most informed action / series of actions in the dataset? For the person that
> made such an action, what was an "economic cost or risk" that they exposed themselves to
> while making this action, or how might this action have ended poorly (beyond simply
> losing more money if they lost the trade)?

Two deliverables, and the second is the one that is not a standard screen:

1. **Name a specific action or series of actions.** Not a method for finding them -- an
   actual timestamp, market(s), size and direction out of this dataset.
2. **Say what it cost the person to do it, beyond the money at risk on the trade.** The
   prompt explicitly rules out "they might lose more money", so max-loss arithmetic does
   not answer this. It is about the second-order consequences of having to show your hand
   to the market in order to get the position on.

#### What we have to work with

| column | use |
| --- | --- |
| `aggressor_buy_flag` | direction. Verified against the book: True -> price is the best ask 94.9% of the time (lifted the offer, bought YES); False -> price is the best bid 95.1% (hit the bid, sold YES). |
| `qty`, `price`, `recv_ts_utc` | size, level, time |
| `native_id` | market, and via `parse_chain_and_strike` the chain and strike |
| `player_id`, `player_name` | **100% null** -- so no trader identity, no per-trader history, no "watch this account going forward" |

The missing identity columns rule out the strongest real-world approach (track a known
account across trades and score its hit rate). Everything below has to work from the
anonymous tape alone.

Also worth stating: the aggressor side is lopsided -- 1,704 sells against 528 buys,
roughly 76/24. A large *buy* is therefore the rarer and more conspicuous event.

#### Step 1 -- aggregate prints into parent actions

A single order does not arrive as a single print. Measured on this dataset: 234 bursts of
2 or more prints land within 50ms of each other in the same market. So ranking raw prints
ranks fragments of orders, not orders.

Group prints into one action when they share a market and a direction and arrive within
50ms of the previous one. Record: market, chain, strike, direction, total qty, n_prints,
n_price_levels, first and last timestamp, and volume-weighted average price.

`n_price_levels` is kept separately because it is the "hit through the stack" signal, and
it is much rarer than the burst count suggests: of those 234 bursts, **only 44 span more
than one price level**. The rest are the exchange filling one order against many resting
orders at the same price, which is not aggression, just mechanics. The largest burst in
the dataset is 28 prints for 41,997 contracts across only 2 price levels.

#### Step 2 -- cluster actions across markets

The question says "action / **series of actions**", which points at one actor expressing
one view in several correlated markets at once. Group actions into a cluster when they
share a direction and arrive within 1 second of each other, across any markets.

This alone is not evidence -- it is common. In this dataset 164 clusters touch more than
one market and 154 touch more than one chain. Simultaneity only becomes a signal in
combination with size, which is what step 3 supplies.

#### Step 3 -- score size against each market's own distribution

A global size threshold is useless here because the markets are on completely different
scales: RFI's median trade is 40 contracts against 7 for TOTAL-3, and a global
3-sigma rule flags 20 trades of which every single one falls in the four busiest markets,
leaving the other 29 markets unable to flag anything at all.

So the fence is computed per market, on that market's own `qty` distribution, using the
Tukey rule `Q3 + 1.5 * IQR`. Median and standard deviation are both unusable: the
distribution is extremely fat-tailed (mean 216, std 1,102, max 27,430) so the large trades
we are hunting inflate the mean and the standard deviation themselves, raising the bar and
hiding the next one. Quartiles are resistant to exactly that.

The Tukey fence is a **screen, not the answer**: it flags 293 of 2,232 trades, about 13%,
across 20 of the 33 markets. Having passed it, actions are then **ranked** by size relative
to their own market's median, so a 1,500-lot in TOTAL-3 (200x its median) can outrank a
3,000-lot in RFI (75x its median).

#### Step 4 -- markout, to separate informed from merely large

Large is not the same as informed. Using `aggressor_buy_flag` for direction, measure how
far the market's mid moved in the trader's favour at +1, +5 and +30 minutes after the
action:

    markout = (mid_after - mid_at_action) * (+1 if bought else -1)

Positive markout means the market moved the trader's way. This is the retrospective
confirmation that an action was informed rather than just big, and it is the reason the
answer can only be identified after the fact -- which is itself worth saying in the
write-up, and is exactly the gap Q4a has to close.

Caveats to apply, both inherited from Q1 and Q2:

- the mid must come from a quote that is actually fresh at the measurement point, not a
  stale one carried forward, so a freshness threshold applies here too
- 66% of trades land inside book-feed gaps of more than 5 seconds, so some markout windows
  will be unmeasurable and should be reported as missing rather than filled

#### Step 5 -- name the winner and write the risk section

The leading candidate on the evidence so far is the cluster at **23:32:07.110 UTC**: 39
prints across 3 markets in 3 different chains, 61,576 contracts in total, roughly 8 minutes
before first pitch. It is four times the size of the next-largest multi-market cluster
(15,017 at 22:18:31). Step 4 either confirms it or replaces it -- the markout decides, not
the size.

For whichever action wins, the economic cost / risk section covers:

1. **Information leakage.** The trade tells everyone watching the tape that someone with
   conviction has taken a side. The edge is worth less the moment it is exercised.
2. **The position cannot be exited.** A 24k position in TOTAL-8 is roughly 25% of
   everything that market traded in six hours. There is no way out at a sensible price, so
   the trader is locked in to settlement. That converts a view about probability into an
   all-or-nothing outcome at $0 or $1 -- they cannot take the win at 0.80 and go home.
3. **The market maker widens or pulls.** Q1 established that one maker appears to quote the
   whole chain off a single fitted distribution. A large aggressive take tells that maker
   they have been adversely selected, so they widen or step away. The informed trader burns
   the edge on the first clip and cannot repeat it.
4. **The trade telegraphs across the whole ladder.** Because the chain is internally
   consistent -- Q1 found zero arbitrage violations anywhere -- repricing one strike forces
   the maker to reprice every correlated strike. Hitting one market leaks the view into ten
   others simultaneously, revealing far more than the trader intended.
5. **Legal and regulatory exposure**, per the cases and the CFTC advisory already written up
   above: if the informational edge came from material non-public information, the downside
   is not a losing trade but disgorgement, fines, exchange suspension, or prosecution.

The max-loss arithmetic (100 contracts at 0.70 risks $70) stays as a single framing
sentence, since the prompt explicitly excludes it as the answer.

#### Open questions to settle before coding

1. **Is 50ms the right window for grouping prints into one order?** It is a guess. Worth
   looking at the actual distribution of within-market inter-print gaps and picking the
   point where the distribution separates, rather than asserting a round number.
2. **Is 1 second the right window for cross-market clustering?** Same issue. The 23:32:07
   cluster spans 39 prints, so the answer partly depends on whether that is one actor or
   several reacting to each other within the same second.
3. **Which markout horizon decides the winner** if +1, +5 and +30 minutes disagree? Worth
   deciding in advance rather than after seeing which one favours a preferred answer.
4. **Do we require the winner to be a cluster, or can a single action win?** The question
   allows either ("action / series of actions").